# 문제 생성 함수 만들기

In [3]:
from glob import glob
from openai import OpenAI
from dotenv import load_dotenv
import os
import base64

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def encode_image(image_path):
    with open(image_path, 'rb') as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [8]:
def image_quiz(image_path):
    base64_image = encode_image(image_path)
    quiz_prompt = '''
    제공한 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어 주세요.
    정답은 (1)~(4) 중 하나만 해당되도록 출제하세요.
    아래는 예시입니다.
    ----- 예시 -----

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 베이커리에서 사람들이 빵을 사는 모습이 담겨 있습니다.
    - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입었습니다.
    - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
    - (4) 점원은 노란색 티셔츠를 입었습니다.

    정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입었습니다.
    (주의: 정답은 (1)~(4) 중 하나만 선택하도록 출제하세요.)
    ======
    '''

    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': quiz_prompt},
                {
                    'type': 'image_url',
                    'image_url': {
                        'url': f'data:image/jpeg;base64,{base64_image}'
                    }
                }
            ]
        }
    ]

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=messages
    )

    return response.choices[0].message.content

In [10]:
q = image_quiz('./images/busan_dive.jpg')
print(q)

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 많은 사람들이 컴퓨터 앞에 앉아 있습니다.
    - (2) 벽에 걸린 큰 스크린에 'DIVE 2024 IN BUSAN'이라는 글이 보입니다.
    - (3) 사람들이 모두 노란색 티셔츠를 입고 있습니다.
    - (4) 천장이 높은 실내 공간입니다.

    정답: (3) 사람들은 모두 노란색 티셔츠를 입고 있지 않습니다.


In [11]:
txt = ''
no = 1
for g in glob('./images/*.jpg'):
    try:
        q = image_quiz(g)
    except Exception as e:
        print(e)
        continue

    divider = f'## 문제 {no}\n\n'
    print(divider)

    txt += divider
    filename = os.path.basename(g)
    txt += f'![image]({filename})\n\n'

    print(q)
    txt += q + '\n\n--------------------\n\n'

    with open('./images/image_quiz.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    no += 1

## 문제 1


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 많은 사람들이 컴퓨터 앞에 앉아 있습니다.
- (2) 배경에 "DIVE 2024 IN BUSAN"이라는 문구가 보입니다.
- (3) 사람들이 비행기에 탑승하기 위해 줄을 서 있습니다.
- (4) 행사는 실내에서 진행되고 있습니다.

정답: (3) 사람들이 비행기에 탑승하기 위해 줄을 서 있는 것이 아니라, 행사에 참여하고 있습니다.
## 문제 2


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 큰 노란색 조형물이 있습니다.
- (2) 'Local Stitch'라는 글자가 적힌 건물이 보입니다.
- (3) 조형물은 붉은색입니다.
- (4) 배경에 여러 층의 건물이 있습니다.

정답: (3) 조형물은 붉은색이 아닌 노란색입니다.
## 문제 3


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 카페 내부는 노란색과 흰색으로 꾸며져 있습니다.
- (2) 의자와 테이블이 비치되어 있는 공간입니다.
- (3) 창문을 통해 바깥의 나무가 보입니다.
- (4) 거대한 피자가 테이블 위에 놓여 있습니다.

정답: (4) 거대한 피자가 테이블 위에 놓여 있는 것이 아니라, 테이블에는 아무것도 놓여 있지 않습니다.
## 문제 4


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 건물은 벽돌로 지어져 있습니다.
- (2) 1층에는 "PERSONAL COFFEE"라는 카페가 있습니다.
- (3) 건물 앞에 주황색 원뿔이 놓여 있습니다.
- (4) 건물은 5층으로 구성되어 있습니다.

정답: (4) 건물은 5층이 아닌 3층으로 구성되어 있습니다.
## 문제 5


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 빵이 진열된 베이커리 가게의 모습이 담겨 있습니다.
- (2) 빵 가격이 표시된 가격표가 보입니다.
- (3) 점원이 노란색 셔츠를 입고 있습니다.
- (4) 여러 종류의 빵이 진열